In [ ]:
%pip install eyepop==3.12.0

In [ ]:
import getpass

EYEPOP_ACCOUNT_ID=input("Enter your Account UUID: ")
EYEPOP_API_KEY=getpass.getpass('Enter your API KEY: ')

In [ ]:
NAMESPACE_PREFIX="XXXXXXXXXXX" # Add your namespace-prefix here

### Define Ability

In [ ]:
from eyepop import EyePopSdk
from eyepop.data.data_types import InferRuntimeConfig, VlmAbilityGroupCreate, VlmAbilityCreate, TransformInto
from eyepop.worker.worker_types import CropForward, ForwardComponent, FullForward, InferenceComponent, Pop
import json


ability_prototypes = [
    VlmAbilityCreate(
        name=f"{NAMESPACE_PREFIX}.find-events.identify-PPE",
        description="Identify if all workers are wearing their helmets",
        worker_release="qwen3-instruct",
        text_prompt="""
          You are a Construction Safety Compliance AI. Your job is to strictly enforce safety helmet regulations for construction personnel. Analyze the image/video to verify that the primary workers in the foreground are actively WEARING safety helmets (hard hats) ON THEIR HEADS.

          Subject Focus & Exclusions: Focus YOUR ANALYSIS on the main group of workers present in the central scene. CRITICAL: If there is a picture-in-picture overlay or a large superimposed person in the corner of the screen, you MUST STILL evaluate the main group of workers on the actual construction site. Do not let one large overlaid figure excuse the rest of the crew. You must check every clearly visible worker standing in the main work area. Completely IGNORE only tiny figures in the deep background or people operating machinery inside cabs.

          PPE Focus (HELMETS ONLY): Your ONLY task is to verify if each primary foreground worker is actively wearing a standard construction hard hat properly on their head.

          The Worn on Head Requirement (CRITICAL): The hard hat MUST be physically on top of the worker's head. If a worker is holding a helmet in their hands, carrying it under their arm, or if a helmet is resting on the ground near them, it DOES NOT COUNT.

          CRITICAL: Look at EVERY person in the center of the frame. It doesn't matter if they are holding a helmet. They must have the hardhat/helmet on their head to mark PPE_On else Fail! Even if you find one person who doesn't have it on then you must FAIL!

          The Hair/Bare Head Rule: If you can see hair texture, a hairline, a hair parting, or the organic shape of a bare human head on any worker, that worker is NOT wearing a helmet.

          Material & Structure Check: Look for the distinct structural features of a manufactured hard hat: a front brim, a smooth synthetic/plastic texture, and a raised crown.
          You must strictly IGNORE all other PPE (face guards, glasses, vests, headphones, earmuffs). CRITICAL: Headphones, earmuffs, and communication headsets are NOT hard hats. Do not mistake the plastic band of a headset for a helmet. If a worker is wearing headphones over a bare head, a beanie, or hair, this is a violation and you MUST output Fail.

          Violation Criteria: If you spot even one primary foreground construction worker whose head is exposed (visible hair or bare head), you must flag the video. It does not matter if they are holding a helmet in their hands—if it is not on their head, it is a violation.

          Output Labels: Fail: Select this label if ANY primary foreground site worker has an exposed head (hair/bare head visible) or is merely holding their helmet rather than wearing it.

          PPE_On: Select this label ONLY if ALL primary foreground site workers are actively wearing structural hard hats ON their heads.

          Output only one label: Fail or PPE_On. No other text.
        """,
        transform_into=TransformInto(),
        config=InferRuntimeConfig(
            max_new_tokens=150,
            fps=8,
            image_size=512
        ),
        is_public=False
    )
]



### Create Ability

In [ ]:
with EyePopSdk.dataEndpoint(api_key=EYEPOP_API_KEY, account_id=EYEPOP_ACCOUNT_ID) as endpoint:
    for ability_prototype in ability_prototypes:
        ability_group = endpoint.create_vlm_ability_group(VlmAbilityGroupCreate(
            name=ability_prototype.name,
            description=ability_prototype.description,
            default_alias_name=ability_prototype.name,
        ))
        ability = endpoint.create_vlm_ability(
            create=ability_prototype,
            vlm_ability_group_uuid=ability_group.uuid,
        )
        ability = endpoint.publish_vlm_ability(
            vlm_ability_uuid=ability.uuid,
            alias_name=ability_prototype.name,
        )
        ability = endpoint.add_vlm_ability_alias(
            vlm_ability_uuid=ability.uuid,
            alias_name=ability_prototype.name,
            tag_name="latest"
        )
        print(f"created ability {ability.uuid} with alias entries {ability.alias_entries}")

### Evalulate on a Single Video

In [ ]:
from pathlib import Path

pop = Pop(components=[
   InferenceComponent(
       ability=f"{NAMESPACE_PREFIX}.find-events.identify-PPE:latest",
   )
])

video_path = ... # Add path to video

with EyePopSdk.workerEndpoint(api_key=EYEPOP_API_KEY) as endpoint:
   endpoint.set_pop(pop)
   sample_video_path = Path(video_path)
   job = endpoint.upload(sample_video_path)
   while result := job.predict():
      print(json.dumps(result, indent=2))

print("Done")